In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import torch

# Load model and tokenizer
model_name = "lukasjanek/xlm-RoBERTa-base-skquad-qa"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

def answer_question(question, context):
    # Tokenize input
    inputs = tokenizer(
        question,
        context,
        return_tensors="pt",
        max_length=512,
        truncation=True,
        padding=True
    )

    # Get model outputs
    with torch.no_grad():
        outputs = model(**inputs)

    # Extract start and end positions
    answer_start = torch.argmax(outputs.start_logits)
    answer_end = torch.argmax(outputs.end_logits) + 1

    # Convert tokens to answer string
    answer_tokens = inputs.input_ids[0][answer_start:answer_end]
    answer = tokenizer.decode(answer_tokens, skip_special_tokens=True)

    # Calculate confidence score
    start_score = torch.softmax(outputs.start_logits, dim=1).max().item()
    end_score = torch.softmax(outputs.end_logits, dim=1).max().item()

    return {
        "answer": answer,
        "start": answer_start.item(),
        "end": answer_end.item(),
        "confidence": (start_score + end_score) / 2
    }

# Example usage
context = "Bratislava je hlavné mesto Slovenska. Má rozlohu 367,6 km štvorcových."
question = "Aká je rozloha Bratislavy?"

result = answer_question(question, context)
print(f"Answer: {result['answer']}")
print(f"Start: {result['start']}")
print(f"End: {result['end']}")
print(f"Confidence: {result['confidence']}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Answer: 367,6 km štvorcových
Start: 20
End: 26
Confidence: 0.7076762020587921


In [ ]:
print("\n--- Testing the trained model on new questions ---")

# Define new test cases or use existing ones
new_test_cases = [
    {
        "context": "Bratislava je hlavné a rozlohou aj počtom obyvateľov najväčšie mesto Slovenska. Je sídlom prezidenta, parlamentu aj vlády samosprávneho kraja, vznikajúceho Európskeho orgánu práce, niekoľkých miestnych centrál medzinárodných organizácií, ako aj viacerých divadiel, múzeí, galérií, univerzít a ďalších inštitúcií. Dominantou mesta je Bratislavský hrad s charakteristickými štyrmi vežičkami a Dóm svätého Martina. Spomedzi moderných stavieb je najvýraznejší Most SNP na Dunaji. V historickom centre je vyhlásená pamiatková rezervácia. V mestskej časti Devín leží zrúcanina rovnomenného hradu a preteká ňou rieka Dunaj.",
        "question": "Aká rieka preteká Bratislavou?"
    },
    {
        "context": "Slovensko vstúpilo do Európskej únie v roku 2004 a je členom NATO od roku 2004. Hlavným mestom je Bratislava.",
        "question": "Kedy Slovensko vstúpilo do Európskej únie?"
    },
    {
        "context": "Bratislava je hlavné mesto Slovenskej republiky. Má rozlohu 368 kilometrov štvorcových a žije v nej približne 475 000 obyvateľov.",
        "question": "Aká je rozloha Bratislavy?"
    },
    {
        "context": "Slovenská republika leží v strednej Európe a je známa svojou bohatou históriou a kultúrou. Medzi významné symboly patrí dvojkríž na trojvrší. Tradičná slovenská kuchyňa zahŕňa bryndzové halušky a kapustnicu. Folklórne tance a piesne sú dôležitou súčasťou kultúrneho dedičstva. Slovensko je tiež krajinou hradov a zámkov, z ktorých mnohé sú prístupné verejnosti. Jeho prírodná krása je charakteristická Vysokými Tatrami a početnými jaskyňami. Mnoho Slovákov sa hlási k rímskokatolíckej cirkvi.",
        "question": "Ktoré jedlo je typické pre slovenskú kuchyňu?"
    },
    {
        "context": "Vysoké Tatry, nachádzajúce sa na severe Slovenska, sú najvyšším pohorím v krajine a sú súčasťou Karpát. Sú obľúbenou destináciou pre turistov a lyžiarov, ponúkajú majestátne štíty ako Gerlachovský štít, ktorý je najvyšší, a prekrásne plesá. Región je chostraňovaný ako národný park a je domovom rôznych druhov živočíchov, vrátane kamzíkov a svišťov. Rozvinutá je tu infraštruktúra pre letnú aj zimnú turistiku, s mnohými značenými chodníkmi a lyžiarskymi strediskami.",
        "question": "Aký je najvyšší štít na Slovensku?"
    },
    {
        "context": "Slovenský jazyk patrí do skupiny západných slovanských jazykov a je úradným jazykom Slovenskej republiky. Je blízky češtine a do určitej miery aj poľštine. Používa latinskú abecedu s rozšíreným počtom diakritických znamienok. Prvé písomné pamiatky slovenského jazyka siahajú do stredoveku. Súčasná podoba spisovného jazyka bola kodifikovaná v 19. storočí. Slovenčina je známa svojou bohatou morfológiou a pomerne flexibilným slovosledom, čo ju robí zaujímavou pre jazykovedcov.",
        "question": "Do ktorej skupiny jazykov patrí slovenčina?"
    },
    {
        "context": "Folklór na Slovensku je mimoriadne pestrý a bohatý, odráža regionálne rozdiely a dlhú históriu. Prejavuje sa v tradičných krojoch, ktoré sú často bohato zdobené a líšia sa farbami a vzormi podľa regiónu. Súčasťou sú aj ľudové tance ako 'čardáš' a 'karička', a spev, ktorý sprevádza rôzne príležitosti – od svadieb po žatvy. Hudobné nástroje ako fujara, gajdy a píšťaly sú neodmysliteľnou súčasťou slovenského folklóru. Mnohé folklórne festivaly, ako napríklad tie vo Východnej, udržujú tieto tradície živé.",
        "question": "Ktoré hudobné nástroje sú typické pre slovenský folklór?"
    }
]

for test in new_test_cases:
    answer = get_answer(test["question"], test["context"])
    print(f"Otázka: {test["question"]}")
    print(f"Odpoveď: {answer}")
    print()


--- Testing the trained model on new questions ---
Otázka: Aká rieka preteká Bratislavou?
Odpoveď:  Dunaj

Otázka: Kedy Slovensko vstúpilo do Európskej únie?
Odpoveď:  v roku 2004

Otázka: Aká je rozloha Bratislavy?
Odpoveď:  368 kilometrov štvorcových

Otázka: Ktoré jedlo je typické pre slovenskú kuchyňu?
Odpoveď:  bryndzové halušky a kapustnicu

Otázka: Aký je najvyšší štít na Slovensku?
Odpoveď:  Gerlachovský štít

Otázka: Do ktorej skupiny jazykov patrí slovenčina?
Odpoveď:  do skupiny západných slovanských jazykov

Otázka: Ktoré hudobné nástroje sú typické pre slovenský folklór?
Odpoveď:  fujara, gajdy a píšťaly



In [ ]:
# Extending the existing test_cases with new examples
new_test_cases_extended = [
    {
        "context": "Najdlhšia rieka na Slovensku je Váh. Pramení v Západných Tatrách a preteká cez celé Slovensko až po Dunaj.",
        "question": "Kde pramení rieka Váh?"
    },
    {
        "context": "Slovenská koruna bola mena Slovenska pred prijatím eura. Euro bolo prijaté 1. januára 2009.",
        "question": "Kedy Slovensko prijalo euro?"
    },
    {
        "context": "Medzi najznámejšie slovenské hrady patrí Spišský hrad, ktorý je zapísaný v zozname svetového dedičstva UNESCO. Oravský hrad je tiež veľmi populárny.",
        "question": "Ktorý slovenský hrad je na zozname UNESCO?"
    },
    {
        "context": "Typickým slovenským jedlom sú bryndzové halušky. Pripravujú sa z zemiakového cesta, bryndze a slaninky.",
        "question": "Z čoho sa pripravujú bryndzové halušky?"
    },
    {
        "context": "Poprad je mesto na severe Slovenska, známe ako vstupná brána do Vysokých Tatier. Má približne 50 000 obyvateľov.",
        "question": "Aké mesto je vstupnou bránou do Vysokých Tatier?"
    }
]

print("\n--- Testovanie pretrénovaného modelu na dodatočných otázkach ---")
for test in new_test_cases_extended:
    answer = get_answer(test["question"], test["context"])
    print(f"Otázka: {test["question"]}")
    print(f"Odpoveď: {answer}")
    print()


--- Testovanie pretrénovaného modelu na dodatočných otázkach ---
Otázka: Kde pramení rieka Váh?
Odpoveď:  Pramení v Západných Tatrách

Otázka: Kedy Slovensko prijalo euro?
Odpoveď:  1. januára 2009

Otázka: Ktorý slovenský hrad je na zozname UNESCO?
Odpoveď:  Spišský hrad

Otázka: Z čoho sa pripravujú bryndzové halušky?
Odpoveď:  z zemiakového cesta, bryndze a slaninky

Otázka: Aké mesto je vstupnou bránou do Vysokých Tatier?
Odpoveď: Poprad

